In [3]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import seaborn as sns

from joblib import Parallel, delayed

from helper_function import fisher_info, solve_pmle, simplex, compute_r_nk
from scipy.stats import norm

sns.set()

def compute_simplex_atoms(C, alpha, thetas, weights):
    k = len(C)
    n = sum(C)

    thetas = np.asarray(thetas, float)
    weights = np.asarray(weights, float)

    rs = compute_r_nk(n, k, alpha, thetas, theta_0=np.min(thetas))
    denom = sum(rs*weights)

    integrand_1 = 1/(1+thetas/n)
    num_1 = sum(rs*integrand_1*weights)
    ratio_1 = 1/n * num_1/denom

    integrand_2 = (1+thetas/k/alpha)/(1+thetas/n)
    num_2 = sum(rs * integrand_2 * weights)
    ratio_2 = k * alpha/n * num_2/denom
    
    simp = np.zeros(k+1)
        
    simp[:k] = (C-alpha) * ratio_1
    simp[k] =  ratio_2
    return simp

def compute_simplex_mc(C, alpha, sampler, M=2000):
    k = len(C)
    n = sum(C)

    rng = np.random.default_rng(0)
    thetas = sampler(rng, M)

    rs = compute_r_nk(n, k, alpha, thetas)
    denom = np.mean(rs)

    integrand_1 = 1/(1+thetas/n)
    num_1 = np.mean(rs*integrand_1)
    ratio_1 = 1/n * num_1/denom

    integrand_2 = (1+thetas/k/alpha)/(1+thetas/n)
    num_2 = np.mean(rs * integrand_2)
    ratio_2 = k * alpha/n * num_2/denom
    
    simp = np.zeros(k+1)
        
    simp[:k] = (C-alpha) * ratio_1
    simp[k] =  ratio_2
    return simp


def rejection_sampling(C, threshold, seed):
    k = len(C)
    N = sum(C)
    assert N/k <= threshold

    count = 0
    np.random.seed(seed)
    while True:
        mask = (np.random.binomial(n=1, p=np.ones(k)/2) == 1)
        if np.mean(C[mask]) <= threshold:
            break
        elif count > 10000:
            raise ValueError(f'count={count}, threshold={threshold}, n/k = {N/k}, n={N}')
        else:
            count +=1
    return mask


def ratio_test(p, p_hat, mle, C, eps, seed):
    if np.isclose(mle, 0) or np.isclose(mle, 1):
        return 0
    else:
        N = sum(C)
        k = len(C)
        threshold = N/(k**0.501)
        mask = rejection_sampling(C=C, threshold=threshold, seed=seed)
        statistics = np.sqrt(k * fisher_info(mle)) * (np.mean(C[mask])-mle) * np.abs(
        np.sum(p[:k][mask])/ np.sum(p_hat[:k][mask]) - 1)
        if statistics > norm.ppf(1 - eps/2):
            return 0
        return 1
        

def run_atoms(alpha, N, seed, thetas, weights):
    rng = np.random.default_rng(seed)
    C = np.zeros(N, dtype=np.int64)
    C[0] = 1
    k = 1
    for _ in range(1, N):
        p = compute_simplex_atoms(C[:k], alpha, thetas, weights)
        assert np.isclose(sum(p), 1)
        s = np.searchsorted(np.cumsum(p), rng.random())
        if s == k:
            k += 1
        C[s] += 1

    mle = solve_pmle(C[:k])
    p = compute_simplex_atoms(C[:k], alpha, thetas, weights)
    p_hat = simplex(C[:k], mle)
    success = ratio_test(p=p, p_hat=p_hat, mle=mle, C=C[:k], eps=0.05, seed=seed)

    return {
        'mle':mle, 
        'alpha': alpha,
        'seed': seed,
        'success': success
        }


def run_mc(alpha, N, seed, sampler):
    rng = np.random.default_rng(seed)
    C = np.zeros(N, dtype=np.int64)
    C[0] = 1
    k = 1

    for _ in range(1, N):
        p = compute_simplex_mc(C[:k], alpha, sampler)
        assert np.isclose(sum(p), 1)
        s = np.searchsorted(np.cumsum(p), rng.random())
        if s == k:
            k += 1
        C[s] += 1

    mle = solve_pmle(C[:k])
    p = compute_simplex_mc(C[:k], alpha, sampler)
    p_hat = simplex(C[:k], mle)
    success = ratio_test(p=p, p_hat=p_hat, mle=mle, C=C[:k], eps=0.05, seed=seed)

    return {
        'mle':mle, 
        'alpha': alpha,
        'seed': seed,
        'success': success
       }


def sampler_uniform(a, b):
    def _s(rng, M):
        return rng.uniform(a, b, size=M)
    return _s

def sampler_gauss():
    def _s(rng, M):
        return np.abs(rng.normal(size=M))
    return _s

def sampler_cauchy():
    def _s(rng, M):
        return np.abs(rng.standard_cauchy(size=M))
    return _s

def sampler_t(df):
    def _s(rng, M):
        return np.abs(rng.standard_t(df=df, size=M))
    return _s


def sampler_dist(dist, a=None, b=None, df=None):
    if dist=='uniform':
        return sampler_uniform(a, b)
    elif dist=='gauss':
        return sampler_gauss()
    elif dist=='cauchy':
        return sampler_cauchy()
    elif dist=='t':
        return sampler_t(df)



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
N = 40000
seeds = range(0, 1000)
dfs = []
alphas = [0.6, 0.7, 0.8]

In [ ]:
thetas = [0]
weights = [1]*len(thetas)
params = [{'alpha':alpha, 'N':N, 'seed':seed, 'thetas':thetas, 'weights':weights}
          for seed in seeds
          for alpha in alphas
          ]
print(len(params))
data = Parallel(n_jobs=20, verbose=12)(
        delayed(run_atoms)(**d) for d in params)
df = pd.DataFrame(data)

df.to_csv(f'data/coverage/atom_seed_{len(seeds)}_N_{N}_alphas_{str(alphas)}_thetas_{str(thetas)}.csv', index=False)


for alpha in alphas:
         print(f'alpha={alpha}, coverage={np.mean(df[df['alpha']==alpha]['success'])}')

dfs.append(df)

300


[Parallel(n_jobs=20)]: Using backend LokyBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done   1 tasks      | elapsed:    0.2s
[Parallel(n_jobs=20)]: Batch computation too fast (0.18412208557128906s.) Setting batch_size=2.
[Parallel(n_jobs=20)]: Done   2 tasks      | elapsed:    0.2s
[Parallel(n_jobs=20)]: Done   3 tasks      | elapsed:    0.2s
[Parallel(n_jobs=20)]: Done   4 tasks      | elapsed:    0.2s
[Parallel(n_jobs=20)]: Done   5 tasks      | elapsed:    0.2s
[Parallel(n_jobs=20)]: Done   6 tasks      | elapsed:    0.2s
[Parallel(n_jobs=20)]: Done   7 tasks      | elapsed:    0.2s
[Parallel(n_jobs=20)]: Done   8 tasks      | elapsed:    0.3s
[Parallel(n_jobs=20)]: Done   9 tasks      | elapsed:    0.3s
[Parallel(n_jobs=20)]: Done  10 tasks      | elapsed:    0.3s
[Parallel(n_jobs=20)]: Done  11 tasks      | elapsed:    0.3s
[Parallel(n_jobs=20)]: Done  12 tasks      | elapsed:    0.3s
[Parallel(n_jobs=20)]: Done  13 tasks      | elapsed:    0.3s
[Parallel(n_jobs=20)]

alpha=0.6, coverage=0.92
alpha=0.7, coverage=0.95
alpha=0.8, coverage=0.95


[Parallel(n_jobs=20)]: Done 300 out of 300 | elapsed:    4.8s finished


In [ ]:
thetas = [0, 3]
weights = [1]*len(thetas)

params = [{'alpha':alpha, 'N':N, 'seed':seed, 'thetas':thetas, 'weights':weights}
          for seed in seeds
          for alpha in alphas
          ]

data = Parallel(n_jobs=20, verbose=12)(
        delayed(run_atoms)(**d) for d in params)
df = pd.DataFrame(data)

df.to_csv(f'data/coverage/atom_seed_{len(seeds)}_N_{N}_alphas_{str(alphas)}_thetas_{str(thetas)}.csv', index=False)


for alpha in alphas:
         print(f'alpha={alpha}, coverage={np.mean(df[df['alpha']==alpha]['success'])}')

dfs.append(df)


[Parallel(n_jobs=20)]: Using backend LokyBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done   1 tasks      | elapsed:    0.2s
[Parallel(n_jobs=20)]: Batch computation too fast (0.19301986694335938s.) Setting batch_size=2.
[Parallel(n_jobs=20)]: Done   2 tasks      | elapsed:    0.2s
[Parallel(n_jobs=20)]: Done   3 tasks      | elapsed:    0.3s
[Parallel(n_jobs=20)]: Done   4 tasks      | elapsed:    0.3s
[Parallel(n_jobs=20)]: Done   5 tasks      | elapsed:    0.3s
[Parallel(n_jobs=20)]: Done   6 tasks      | elapsed:    0.3s
[Parallel(n_jobs=20)]: Done   7 tasks      | elapsed:    0.3s
[Parallel(n_jobs=20)]: Done   8 tasks      | elapsed:    0.3s
[Parallel(n_jobs=20)]: Done   9 tasks      | elapsed:    0.3s
[Parallel(n_jobs=20)]: Done  10 tasks      | elapsed:    0.4s
[Parallel(n_jobs=20)]: Done  11 tasks      | elapsed:    0.4s
[Parallel(n_jobs=20)]: Done  12 tasks      | elapsed:    0.4s
[Parallel(n_jobs=20)]: Done  13 tasks      | elapsed:    0.4s
[Parallel(n_jobs=20)]

alpha=0.6, coverage=0.95
alpha=0.7, coverage=0.89
alpha=0.8, coverage=0.98


[Parallel(n_jobs=20)]: Done 300 out of 300 | elapsed:    5.6s finished


In [ ]:
dist = 'uniform'
sampler = sampler_dist(dist, a=0, b=3)

params = [{'alpha':alpha, 'N':N, 'seed':seed, 'sampler':sampler}
          for seed in seeds
          for alpha in alphas
          ]

data = Parallel(n_jobs=20, verbose=12)(
        delayed(run_mc)(**d) for d in params)

df = pd.DataFrame(data)

df.to_csv(f'data/coverage/uniform_seed_{len(seeds)}_N_{N}_alphas_{str(alphas)}.csv', index=False)

for alpha in alphas:
         print(f'alpha={alpha}, coverage={np.mean(df[df['alpha']==alpha]['success'])}')

dfs.append(df)

[Parallel(n_jobs=20)]: Using backend LokyBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done   1 tasks      | elapsed:    0.9s
[Parallel(n_jobs=20)]: Done   2 tasks      | elapsed:    0.9s
[Parallel(n_jobs=20)]: Done   3 tasks      | elapsed:    0.9s
[Parallel(n_jobs=20)]: Done   4 tasks      | elapsed:    1.0s
[Parallel(n_jobs=20)]: Done   5 tasks      | elapsed:    1.0s
[Parallel(n_jobs=20)]: Done   6 tasks      | elapsed:    1.0s
[Parallel(n_jobs=20)]: Done   7 tasks      | elapsed:    1.0s
[Parallel(n_jobs=20)]: Done   8 tasks      | elapsed:    1.0s
[Parallel(n_jobs=20)]: Done   9 tasks      | elapsed:    1.0s
[Parallel(n_jobs=20)]: Done  10 tasks      | elapsed:    1.1s
[Parallel(n_jobs=20)]: Done  11 tasks      | elapsed:    1.1s
[Parallel(n_jobs=20)]: Done  12 tasks      | elapsed:    1.1s
[Parallel(n_jobs=20)]: Done  13 tasks      | elapsed:    1.2s
[Parallel(n_jobs=20)]: Done  14 tasks      | elapsed:    1.2s
[Parallel(n_jobs=20)]: Done  15 tasks      | elapsed:  

alpha=0.6, coverage=0.91
alpha=0.7, coverage=0.94
alpha=0.8, coverage=0.98


[Parallel(n_jobs=20)]: Done 300 out of 300 | elapsed:   15.0s finished


In [ ]:
dist = 'gauss'
sampler = sampler_dist(dist=dist)

params = [{'alpha':alpha, 'N':N, 'seed':seed, 'sampler':sampler}
          for seed in seeds
          for alpha in alphas
          ]

data = Parallel(n_jobs=20, verbose=12)(
        delayed(run_mc)(**d) for d in params)

df = pd.DataFrame(data)
df.to_csv(f'data/coverage/gauss_seed_{len(seeds)}_N_{N}_alphas_{str(alphas)}.csv', index=False)

for alpha in alphas:
         print(f'alpha={alpha}, coverage={np.mean(df[df['alpha']==alpha]['success'])}')

dfs.append(df)


[Parallel(n_jobs=20)]: Using backend LokyBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done   1 tasks      | elapsed:    0.8s
[Parallel(n_jobs=20)]: Done   2 tasks      | elapsed:    0.9s
[Parallel(n_jobs=20)]: Done   3 tasks      | elapsed:    0.9s
[Parallel(n_jobs=20)]: Done   4 tasks      | elapsed:    0.9s
[Parallel(n_jobs=20)]: Done   5 tasks      | elapsed:    0.9s
[Parallel(n_jobs=20)]: Done   6 tasks      | elapsed:    0.9s
[Parallel(n_jobs=20)]: Done   7 tasks      | elapsed:    0.9s
[Parallel(n_jobs=20)]: Done   8 tasks      | elapsed:    1.0s
[Parallel(n_jobs=20)]: Done   9 tasks      | elapsed:    1.0s
[Parallel(n_jobs=20)]: Done  10 tasks      | elapsed:    1.0s
[Parallel(n_jobs=20)]: Done  11 tasks      | elapsed:    1.0s
[Parallel(n_jobs=20)]: Done  12 tasks      | elapsed:    1.0s
[Parallel(n_jobs=20)]: Done  13 tasks      | elapsed:    1.0s
[Parallel(n_jobs=20)]: Done  14 tasks      | elapsed:    1.0s
[Parallel(n_jobs=20)]: Done  15 tasks      | elapsed:  

alpha=0.6, coverage=0.96
alpha=0.7, coverage=0.95
alpha=0.8, coverage=0.97


[Parallel(n_jobs=20)]: Done 300 out of 300 | elapsed:   14.8s finished
